# Cleanup Target Registered Models

Deletes registered model versions and model entries in the target catalog. Does not delete files in the import volume. Use to re-run import with a clean registry (same model names, no suffix).

## Configuration

Read widget parameters: target catalog, schema, and comma-separated model names to delete.

In [ ]:
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("target_schema", "default", "Target schema")
dbutils.widgets.text("model_names", "", "Model names comma-separated")
dbutils.widgets.text("experiment_prefix", "migration_", "Prefix for the per-model migration experiment ('' = no prefix)")

TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip()
model_names_str = dbutils.widgets.get("model_names").strip()
EXPERIMENT_PREFIX = dbutils.widgets.get("experiment_prefix").strip()
if not TARGET_CATALOG or not model_names_str:
    raise ValueError("Set target_catalog and model_names")
MODELS = [m.strip() for m in model_names_str.split(",") if m.strip()]
print(f"Catalog: {TARGET_CATALOG}.{TARGET_SCHEMA}")
print(f"Models: {MODELS}")

## Delete registered models and experiments

For each model: deletes all versions, the registered model entry, the associated migration runs, and the migration experiment. Skips gracefully if resources do not exist.

In [ ]:
import mlflow
from mlflow import MlflowClient
from mlflow.entities import ViewType
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
try:
    user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
except Exception:
    try:
        user_name = spark.conf.get("spark.databricks.userContext.userName")
    except Exception:
        user_name = "unknown"

KNOWN_ALIASES = ["Champion", "Challenger", "Shadow"]

def is_not_found(e):
    msg = str(e).lower()
    return "resource_does_not_exist" in msg or "does not exist" in msg or "not found" in msg

def get_all_versions(full_name):
    all_versions = []
    page_token = None
    while True:
        page = client.search_model_versions(
            filter_string=f"name='{full_name}'",
            max_results=200,
            page_token=page_token,
        )
        all_versions.extend(page)
        page_token = getattr(page, "token", None)
        if not page_token:
            break
    return all_versions

summary = []
for model_name in MODELS:
    full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{model_name}"
    print(f"\n--- {full_name} ---")

    try:
        versions = get_all_versions(full_name)
    except Exception as e:
        if is_not_found(e):
            print(f"  Not found, skipping")
            summary.append((model_name, 0))
            continue
        raise

    if not versions:
        print(f"  No versions found")

    for alias in KNOWN_ALIASES:
        try:
            client.delete_registered_model_alias(full_name, alias)
            print(f"  Removed alias: {alias}")
        except Exception:
            pass

    deleted_versions = 0
    for v in versions:
        try:
            client.delete_model_version(name=full_name, version=v.version)
            deleted_versions += 1
        except Exception as e:
            if not is_not_found(e):
                print(f"  WARN v{v.version} delete failed: {e}")
        if getattr(v, "run_id", None):
            try:
                client.delete_run(v.run_id)
            except Exception:
                pass
    print(f"  Deleted {deleted_versions}/{len(versions)} version(s)")

    try:
        client.delete_registered_model(full_name)
        print(f"  Deleted registered model")
    except Exception as e:
        if not is_not_found(e):
            print(f"  WARN model delete failed: {e}")

    exp_name = f"/Users/{user_name}/{EXPERIMENT_PREFIX}{model_name}"
    exp = client.get_experiment_by_name(exp_name)
    if exp:
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            filter_string="",
            run_view_type=ViewType.ALL,
        )
        for run in runs:
            try:
                client.delete_run(run.info.run_id)
            except Exception:
                pass
        try:
            client.delete_experiment(exp.experiment_id)
            print(f"  Deleted experiment {exp_name} ({len(runs)} run(s))")
        except Exception:
            pass

    summary.append((model_name, deleted_versions))

print("\n" + "=" * 50)
print("CLEANUP SUMMARY")
for name, count in summary:
    print(f"  {TARGET_CATALOG}.{TARGET_SCHEMA}.{name}: {count} version(s) deleted")
print("=" * 50)